# Eksperiment 2-4 — Forma (EMA 50%)

Predikcija ishoda (Away Win / Draw / Home Win) koristeci SAMO formu tima racunatu kao Exponential Moving Average sa alpha=50% (`home_form_ema50`, `away_form_ema50`).

In [6]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix


class MLPClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 3)
        )

    def forward(self, x):
        return self.network(x)


def run_experiment(
    X_train_tensor,
    y_train_tensor,
    X_test_tensor,
    y_test_tensor,
    num_runs=10,
    num_epochs=100,
    batch_size=16,
    learning_rate=0.001
):
    accuracies = []
    class_names = ["Away Win", "Draw", "Home Win"]
    input_dim = X_train_tensor.shape[1]

    for run in range(num_runs):
        print(f"Run {run + 1}/{num_runs}")

        seed = 42 + run
        torch.manual_seed(seed)
        np.random.seed(seed)

        train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True
        )

        model = MLPClassifier(input_dim=input_dim)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate
        )

        for epoch in range(num_epochs):
            model.train()

            for X_batch, y_batch in train_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        model.eval()

        with torch.no_grad():
            test_outputs = model(X_test_tensor)
            y_pred = torch.argmax(test_outputs, dim=1)

        y_pred_np = y_pred.numpy()
        y_test_np = y_test_tensor.numpy()

        acc = accuracy_score(y_test_np, y_pred_np)
        accuracies.append(acc)

        print(f"Accuracy: {acc:.4f}")
        print()

    mean_accuracy = np.mean(accuracies)

    print("=================================")
    print(f"Mean accuracy: {mean_accuracy:.4f}")
    print("=================================")

    print("Classification report for last run:")
    print(classification_report(
        y_test_np,
        y_pred_np,
        target_names=class_names
    ))

    cm = confusion_matrix(y_test_np, y_pred_np)

    cm_df = pd.DataFrame(
        cm,
        index=[f"True {name}" for name in class_names],
        columns=[f"Pred {name}" for name in class_names]
    )

    print("Confusion matrix for last run:")
    print(cm_df)

    return accuracies, mean_accuracy

## Priprema podataka

Uzimamo samo `home_form_ema50` i `away_form_ema50` kao feature, target je `result` (-1/0/1) mapiran u klase 0/1/2 (Away/Draw/Home), isti redosled kao `class_names` iznad.

In [7]:
FEATURE_COLS = ["home_form_ema50", "away_form_ema50"]

data = pd.read_csv("../data/Processed_Project_Data/fifa_and_elo_with_ema.csv")
data = data.dropna(subset=FEATURE_COLS + ["result"])

X = data[FEATURE_COLS].values.astype(np.float32)
y = (data["result"].values + 1).astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)



## Trening (10 run-ova)

In [5]:
accuracies, mean_accuracy = run_experiment(
    X_train_tensor, y_train_tensor, X_test_tensor, y_test_tensor
)

Run 1/10
Accuracy: 0.5225

Run 2/10
Accuracy: 0.5180

Run 3/10
Accuracy: 0.5225

Run 4/10
Accuracy: 0.5315

Run 5/10
Accuracy: 0.5315

Run 6/10
Accuracy: 0.5315

Run 7/10
Accuracy: 0.5315

Run 8/10
Accuracy: 0.5270

Run 9/10
Accuracy: 0.5225

Run 10/10
Accuracy: 0.5270

Mean accuracy: 0.5266
Classification report for last run:
              precision    recall  f1-score   support

    Away Win       0.41      0.38      0.39        61
        Draw       0.00      0.00      0.00        47
    Home Win       0.57      0.82      0.67       114

    accuracy                           0.53       222
   macro avg       0.33      0.40      0.35       222
weighted avg       0.40      0.53      0.45       222

Confusion matrix for last run:
               Pred Away Win  Pred Draw  Pred Home Win
True Away Win             23          0             38
True Draw                 13          0             34
True Home Win             20          0             94


/Users/daniloivanisevic/miniconda3/envs/thestrokes/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/daniloivanisevic/miniconda3/envs/thestrokes/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/daniloivanisevic/miniconda3/envs/thestrokes/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this beha

## Cuvanje rezultata

In [ ]:
results_df = pd.DataFrame({"run": range(1, len(accuracies) + 1), "accuracy": accuracies})
results_df.to_csv("results/exp_2_4_ema50.csv", index=False)
results_df